# Ingest: raw match JSON to one table
Every game of every kept player becomes one row with all 358 fields. No cleaning here.

In [2]:
import json
from pathlib import Path
import pandas as pd

# The notebook runs from notebooks/, so ../ goes up to pipeline1-lol-rank/
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

## Kept players
The 1,361 players with at least 8 recent ranked games, from the three regional pulls.

In [3]:
# Stack used_players_americas.csv, used_players_asia.csv, used_players_europe.csv into one table
kept_players = pd.concat(
    [pd.read_csv(csv_path) for csv_path in sorted(RAW_DATA_DIR.glob("used_players_*.csv"))],
    ignore_index=True,
)
print(kept_players.shape)
print(kept_players["server"].value_counts())

(1361, 4)
server
euw1    359
kr      345
na1     332
eun1    325
Name: count, dtype: int64


## One game, one player
A single match file first: 10 participants, only one of them is the player I sampled.

In [4]:
# Take the first kept player
player = kept_players.iloc[0]

# Open their match ID list: data/raw/match_ids/{server}_{puuid}.json
match_ids_path = RAW_DATA_DIR / "match_ids" / f"{player['server']}_{player['puuid']}.json"
player_match_ids = json.load(open(match_ids_path))
print(player["server"], player["tier"], len(player_match_ids), "games")

# Open their first game
match_data = json.load(open(RAW_DATA_DIR / "matches" / f"{player_match_ids[0]}.json", encoding="utf-8"))

# Print the 10 participants; is_sampled_player is True only for our player
for participant in match_data["info"]["participants"]:
    is_sampled_player = participant["puuid"] == player["puuid"]
    print(is_sampled_player, participant["championName"], participant["teamPosition"],
          participant["teamId"], participant["win"])

na1 IRON 15 games
False Gnar TOP 100 True
False Rammus JUNGLE 100 True
False TwistedFate MIDDLE 100 True
False Kaisa BOTTOM 100 True
False Velkoz UTILITY 100 True
False Zaahen TOP 200 False
True Volibear JUNGLE 200 False
False Vex MIDDLE 200 False
False Tristana BOTTOM 200 False
False Nautilus UTILITY 200 False


In [5]:
# Pull out the sampled player's block from the 10 participants
sampled_participant = next(
    participant for participant in match_data["info"]["participants"]
    if participant["puuid"] == player["puuid"]
)

# Flatten nested fields (challenges, perks, ...) into dotted column names
sampled_participant_flat = pd.json_normalize(sampled_participant)
print(sampled_participant_flat.shape)

pd.set_option("display.max_rows", 400)
sampled_participant_flat.T   # .T shows one field per row so you can scroll

(1, 296)


,0
PlayerScore0,0
PlayerScore1,0
PlayerScore10,0
PlayerScore11,0
PlayerScore2,0
PlayerScore3,0
PlayerScore4,0
PlayerScore5,0
PlayerScore6,0
PlayerScore7,0


## From match file to one row
Player fields, match fields (`match.`) and team fields (`team.`).

In [6]:
def build_game_row(match_data, player_puuid):
    """One match + one player's puuid -> one row with every field, no selection."""
    match_info = match_data["info"]

    # Find the sampled player among the 10 participants
    sampled_participant = next(
        (participant for participant in match_info["participants"]
         if participant["puuid"] == player_puuid),
        None,
    )
    if sampled_participant is None:
        return None

    # 1. Every field of the sampled player (nested ones flattened with dots)
    game_row = pd.json_normalize(sampled_participant).iloc[0].to_dict()

    # 2. Every match-level field, prefixed "match."
    game_row["match.matchId"] = match_data["metadata"]["matchId"]
    for field_name, field_value in match_info.items():
        if field_name not in ("participants", "teams"):
            game_row["match." + field_name] = field_value

    # 3. The sampled player's team: objectives and bans, prefixed "team."
    sampled_team = next(
        team for team in match_info["teams"]
        if team["teamId"] == sampled_participant["teamId"]
    )
    sampled_team_flat = pd.json_normalize(sampled_team).iloc[0].to_dict()
    for field_name, field_value in sampled_team_flat.items():
        game_row["team." + field_name] = field_value

    return game_row

## Build games_raw.csv
Loop over every kept player and game, then attach their ladder info (`ladder.`).

In [7]:
ladder_players = pd.read_csv(RAW_DATA_DIR / "players_sampled.csv").drop_duplicates("puuid")

all_game_rows = []
missing_match_files = 0     # match ID listed, but no file in data/raw/matches/
player_not_in_match = 0     # file exists, but the puuid isn't among the 10 participants

for player_index, player in kept_players.iterrows():
    match_ids_path = RAW_DATA_DIR / "match_ids" / f"{player['server']}_{player['puuid']}.json"
    player_match_ids = json.load(open(match_ids_path))

    for match_id in player_match_ids:
        match_file_path = RAW_DATA_DIR / "matches" / f"{match_id}.json"
        if not match_file_path.exists():
            missing_match_files += 1
            continue

        match_data = json.load(open(match_file_path, encoding="utf-8"))
        game_row = build_game_row(match_data, player["puuid"])
        if game_row is None:
            player_not_in_match += 1
            continue
        all_game_rows.append(game_row)

    if player_index % 100 == 0:
        print(player_index, "players done")

games_table = pd.DataFrame(all_game_rows)

# Attach every column from players_sampled.csv, prefixed "ladder."
games_table = games_table.merge(
    ladder_players.add_prefix("ladder."),
    left_on="puuid", right_on="ladder.puuid",
)
games_table = games_table.drop(columns="ladder.puuid")

games_table.to_csv(PROCESSED_DATA_DIR / "games_raw.csv", index=False)
print(games_table.shape, "| missing match files:", missing_match_files,
      "| player not in match:", player_not_in_match)

0 players done
100 players done
200 players done
300 players done
400 players done
500 players done
600 players done
700 players done
800 players done
900 players done
1000 players done
1100 players done
1200 players done
1300 players done
(19824, 358) | missing match files: 0 | player not in match: 1


## Checks
Games per tier, queue (all 420, ranked solo), game length.

In [8]:
print(games_table["ladder.tier"].value_counts())
print(games_table["match.queueId"].value_counts())
print(games_table["match.gameDuration"].describe())

ladder.tier
MASTER         2700
GRANDMASTER    2700
CHALLENGER     2700
DIAMOND        2366
IRON           1710
PLATINUM       1672
EMERALD        1539
BRONZE         1526
SILVER         1466
GOLD           1445
Name: count, dtype: int64
match.queueId
420    19824
Name: count, dtype: int64
count    19824.000000
mean      1658.202028
std        483.238886
min         30.000000
25%       1416.000000
50%       1681.000000
75%       1947.000000
max       6447.000000
Name: match.gameDuration, dtype: float64


## Column list
Saves columns_raw.csv, the base for the column dictionary.

In [9]:
column_summary = pd.DataFrame({
    "column": games_table.columns,
    "dtype": games_table.dtypes.astype(str).values,
    "non_null_count": games_table.notna().sum().values,
    "example_value": games_table.iloc[0].astype(str).str[:40].values,
})
column_summary.to_csv(PROCESSED_DATA_DIR / "columns_raw.csv", index=False)
print(len(column_summary), "columns")

358 columns


## What the item IDs mean
Item names from Riot's Data Dragon, to read the trinket slot (item6) and the role quest slot (roleBoundItem).

In [ ]:
import requests

DDRAGON_VERSION = "16.20.1"   # latest patch; my games run on 16.x patches
item_json = requests.get(
    f"https://ddragon.leagueoflegends.com/cdn/{DDRAGON_VERSION}/data/en_US/item.json"
).json()["data"]
item_names = {int(item_id): item["name"] for item_id, item in item_json.items()}

# What trinkets players ended with
print(games_table["item6"].map(item_names).value_counts(dropna=False))

# What the role quest slot holds, per role
print(games_table.groupby("teamPosition")["roleBoundItem"].apply(
    lambda ids: ids.map(item_names).value_counts().head(3)))

item6
Oracle Lens            7911
Stealth Ward           7434
Farsight Alteration    4339
Scarecrow Effigy         68
NaN                      41
Eye of the Herald        31
Name: count, dtype: int64
teamPosition                                            
              Support Quest                                    1
              Mid Lane Quest                                   1
BOTTOM        Berserker's Greaves                           1536
              Gluttonous Greaves                             768
              Boots of Swiftness                             403
JUNGLE        Jungle Quest Reward                           3429
              Jungle Quest                                   231
MIDDLE        Mid Lane Quest Reward                         4018
              Mid Lane Quest                                 119
TOP           Unleashed Teleport (Top Lane Quest Reward)    2465
              Top Lane Quest Reward                         1165
              Top Lane Quest